# Iris Endpoint Setup In SageMaker Notebook
Run each cell in order. This notebook trains a model and deploys a real-time SageMaker endpoint for your website.

In [ ]:
import os
import json
from datetime import datetime

import boto3
from botocore.exceptions import ClientError
import sagemaker
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

In [ ]:
boto_sess = boto3.Session()
region = boto_sess.region_name
role = sagemaker.get_execution_role()
account_id = boto_sess.client("sts").get_caller_identity()["Account"]

bucket_name = f"sagemaker-{region}-{account_id}"
s3_client = boto_sess.client("s3", region_name=region)

try:
    s3_client.head_bucket(Bucket=bucket_name)
except ClientError as e:
    code = e.response.get("Error", {}).get("Code", "")
    if code in ["404", "NoSuchBucket", "NotFound"]:
        if region == "us-east-1":
            s3_client.create_bucket(Bucket=bucket_name)
        else:
            s3_client.create_bucket(
                Bucket=bucket_name,
                CreateBucketConfiguration={"LocationConstraint": region},
            )
    else:
        raise

session = sagemaker.Session(boto_session=boto_sess, default_bucket=bucket_name)

endpoint_name = f"iris-endpoint-{datetime.utcnow().strftime('%Y%m%d%H%M%S')}"
train_instance_type = "ml.m5.large"
inference_instance_type = "ml.m5.large"

print("Region:", region)
print("Role:", role)
print("Account:", account_id)
print("Bucket:", bucket_name)
print("Endpoint Name:", endpoint_name)

In [ ]:
# Create the training script file
training_script = """import os
import json
import joblib

from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def main():
    iris = load_iris()
    x_train, x_test, y_train, y_test = train_test_split(
        iris.data,
        iris.target,
        test_size=0.2,
        random_state=42,
        stratify=iris.target,
    )

    model = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=400, random_state=42)),
        ]
    )

    model.fit(x_train, y_train)
    preds = model.predict(x_test)
    acc = accuracy_score(y_test, preds)
    print(f"Validation accuracy: {acc:.4f}")

    model_dir = os.environ.get("SM_MODEL_DIR", "/opt/ml/model")
    os.makedirs(model_dir, exist_ok=True)

    joblib.dump(model, os.path.join(model_dir, "model.joblib"))

    with open(os.path.join(model_dir, "target_names.json"), "w", encoding="utf-8") as f:
        json.dump(iris.target_names.tolist(), f)


if __name__ == "__main__":
    main()
"""

# Write the training script to file
with open("train.py", "w") as f:
    f.write(training_script)

print("✓ Created train.py successfully")

In [ ]:
# Create the inference script file
inference_script = """import json
import os

import joblib
import numpy as np


class IrisModel:
    def __init__(self, model, class_names):
        self.model = model
        self.class_names = class_names


def model_fn(model_dir):
    model = joblib.load(os.path.join(model_dir, "model.joblib"))

    class_names_path = os.path.join(model_dir, "target_names.json")
    if os.path.exists(class_names_path):
        with open(class_names_path, "r", encoding="utf-8") as f:
            class_names = json.load(f)
    else:
        class_names = ["setosa", "versicolor", "virginica"]

    return IrisModel(model=model, class_names=class_names)


def input_fn(request_body, request_content_type):
    if "application/json" not in request_content_type:
        raise ValueError("Only application/json is supported")

    payload = json.loads(request_body)

    if isinstance(payload, dict):
        payload = payload.get("data")

    if not isinstance(payload, list):
        raise ValueError("Input must be a list of numbers or a list of rows")

    if payload and all(isinstance(v, (int, float)) for v in payload):
        arr = np.array([payload], dtype=np.float64)
    else:
        arr = np.array(payload, dtype=np.float64)

    if arr.ndim != 2 or arr.shape[1] != 4:
        raise ValueError("Input must have shape [n,4]")

    return arr


def predict_fn(input_data, model_bundle):
    preds = model_bundle.model.predict(input_data)
    return [model_bundle.class_names[int(p)] for p in preds]


def output_fn(prediction, accept):
    if "application/json" not in accept:
        raise ValueError("Only application/json is supported")

    if len(prediction) == 1:
        response = {"prediction": prediction[0]}
    else:
        response = {"prediction": prediction}

    return json.dumps(response), "application/json"
"""

# Write the inference script to file
with open("inference.py", "w") as f:
    f.write(inference_script)

print("✓ Created inference.py successfully")

In [ ]:
output_path = f"s3://{bucket_name}/iris-training-output"

estimator = SKLearn(
    entry_point="train.py",
    source_dir=".",
    framework_version="1.2-1",
    py_version="py3",
    role=role,
    instance_count=1,
    instance_type=train_instance_type,
    output_path=output_path,
    sagemaker_session=session,
)

print("Training output path:", output_path)
estimator.fit(wait=True, logs=True)

In [ ]:
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type=inference_instance_type,
    endpoint_name=endpoint_name,
    entry_point="inference.py",
    source_dir=".",
)

predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

print("Deployed endpoint:", endpoint_name)

In [ ]:
test_input = [5.1, 3.5, 1.4, 0.2]
result = predictor.predict(test_input)
print(result)

## Connect Your Website
Copy the printed endpoint name and set it in your website backend environment variable SAGEMAKER_ENDPOINT_NAME.

In [ ]:
# Optional cleanup to avoid charges after testing
# predictor.delete_endpoint()
# predictor.delete_model()